# 🧪 Text-to-GraphQL: End-to-End Pipeline Testing

This notebook tests the full flow: **User Query -> Agent Supervisor -> Planner -> RAG Retrieval -> GraphQL Execution -> Final Answer**.

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("/Users/hiep.t.nguyen/Desktop/Workspace/hiep/tcb-text-to-graphql")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import settings
from src.context.ingest import ingest_metadata
from src.context.retriever import LlamaContextRetriever
from src.tools.context_retrieval import set_retriever
from src.graphql_facade import data_loader
from src.agents.supervisor import chat

import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logger = logging.getLogger(__name__)

/Users/hiep.t.nguyen/Desktop/Workspace/hiep/tcb-text-to-graphql/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Pipeline Initialization
Loading data, ingesting metadata, and setting up the global retriever.

In [2]:
# 1. Load sample JSON data for GraphQL
data_loader.load_data()
print("✅ Sample data loaded (JSON files)")

# 2. Initialize RAG Index & Load Nodes (from Chroma automatically)
index, nodes = ingest_metadata()
retriever = LlamaContextRetriever(index, nodes=nodes)
set_retriever(retriever)
print(f"✅ RAG Index initialized with {len(nodes)} nodes (Hybrid Fusion ready)")

INFO:src.graphql_facade.data_loader:Loaded 3 customer info records
INFO:src.graphql_facade.data_loader:Loaded 9 finance profile records
✅ Sample data loaded (JSON files)
INFO:src.context.ingest:Found existing metadata in ChromaDB collection 'llama_metadata'. Skipping ingestion.
DEBUG:bm25s:Building index from IDs objects
INFO:src.context.retriever:BM25 retriever initialized with 674 nodes
✅ RAG Index initialized with 674 nodes (Hybrid Fusion ready)


## 2. End-to-End Chat Testing
Testing complex queries that require planning, retrieval, and GraphQL execution.

In [3]:
def test_query(query: str):
    print(f"\n{'='*50}")
    print(f"USER QUERY: {query}")
    print(f"{'='*50}\n")
    
    # Note: This calls the LangGraph supervisor which uses the Planner tool
    response = chat(query, session_id="test-e2e")
    
    print(f"\nASSISTANT RESPONSE:\n{response}")

# Test 1: Customer Overview
test_query("Cho tôi tổng quan khách hàng Nguyen van A")


USER QUERY: Cho tôi tổng quan khách hàng Nguyen van A

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated queries:
Dưới đây là 3 biến thể truy vấn tìm kiếm khác nhau cho câu hỏi "Cho tôi tổng quan khách hàng Nguyen van A" với các góc nhìn khác nhau:
1. **Sử dụng từ đồng nghĩa và mở rộng phạm vi thông tin**
*Truy vấn:*
"Vui lòng cung cấp hồ sơ chi tiết và lịch sử giao dịch của khách hàng Nguyễn Văn A, bao gồm thông tin cá nhân, tài khoản hiện có và các sản phẩm tài chính đang sử dụng."
2. **Thu hẹp phạm vi, tập trung vào dữ liệu tài chính và trạng thái tài khoản**
*Truy vấn:*
"Cho tôi báo cáo tình trạng tài khoản và số dư hiện tại của khách hàng Nguyễn Văn A, cùng với 

In [4]:
# Test 2: Specific Financial Field
test_query("CASA hiện tại của Cus1 là bao nhiêu?")


USER QUERY: CASA hiện tại của Cus1 là bao nhiêu?

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated queries:
Dưới đây là 3 biến thể truy vấn tìm kiếm khác nhau cho câu hỏi "CASA hiện tại của Cus1 là bao nhiêu?" với các góc nhìn khác nhau:
1. **Sử dụng từ đồng nghĩa và mở rộng phạm vi**
*Truy vấn:*
"Số dư tài khoản thanh toán và tiết kiệm (CASA) hiện tại của khách hàng mã Cus1 là bao nhiêu?"
*Góc nhìn:* Mở rộng câu hỏi để bao gồm cả tài khoản thanh toán và tiết kiệm, giúp tìm kiếm thông tin tổng hợp về CASA.
2. **Thu hẹp phạm vi và tập trung vào thời điểm cụ thể**
*Truy vấn:*
"Số dư tài khoản CASA của khách hàng Cus1 tính đến ngày hôm nay là bao nhiêu?"
*Góc nhìn:* Tập trung vào số dư tại một thời điểm cụ thể, giúp truy vấn dữ liệu theo thời gian.
3. **Xem 

In [5]:
# Test 3: Multiple Information Points
test_query("Top 3 NBO và AUM 3 tháng của khách hàng Pham Quang Minh là gì?")


USER QUERY: Top 3 NBO và AUM 3 tháng của khách hàng Pham Quang Minh là gì?

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated queries:
Dưới đây là 3 biến thể truy vấn tìm kiếm khác nhau cho câu hỏi gốc, mỗi biến thể khai thác một góc nhìn riêng biệt:
---
**1. Biến thể sử dụng từ đồng nghĩa và mở rộng phạm vi thời gian (góc nhìn rộng hơn):**
*Truy vấn:*
"Cho biết tổng giá trị NBO (New Business Orders) và tổng tài sản quản lý (AUM) trong 3 tháng gần nhất của khách hàng Phạm Quang Minh, bao gồm cả các hợp đồng mới và gia hạn."
*Giải thích:*
- Sử dụng từ đồng nghĩa như "tổng giá trị" thay cho "top 3".
- Mở rộng ý nghĩa để bao gồm cả hợp đồng mới và gia hạn, không chỉ đơn thuần là top 3 giao dịch.
- Giữ nguyên khoảng thời gian 3 tháng.
---
**2. Biến thể tập tru